# **04 - Conformal Prediction Application**

## **Summary**

---
## **Table of Contents**

---
## **Objective**

## <a id ='data-info'></a> **Dataset Info**
* **Source:** UCI Machine Learning Repository
* **Title:** Appliances Energy Prediction
* **Time Range:** 11th January 2016 - 27th May 2016
* **Primary Target Variable:** `Appliances` (Watt-hour consumed)

## **Initial Setup**

In [ ]:
import pandas as pd
from pathlib import Path
import sys
from sklearn.model_selection import train_test_split
import joblib
import numpy as np
from aci.tracker import ACITracker
import matplotlib.pyplot as plt

# Add the root directory (one level up from notebooks/) to Python's path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.preprocess import preprocess_energy, feature_engineering, arimax_preprocess

In [ ]:
PROJECT_ROOT = Path.cwd().parent

# Import csv file
energy = pd.read_csv(f'{PROJECT_ROOT}/data/raw/energydata_complete.csv')

In [ ]:
# Data preprocessing
energy = preprocess_energy(energy)

# Form feature through feature engineering
energy = feature_engineering(energy)

# Preprocess for ARIMAX
X, y = arimax_preprocess(energy)

# Train-calibration-test split (70-15-15)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle= False
)

X_cal, X_test, y_cal, y_test = train_test_split(
    X_test, y_test, shuffle= False
)

## **Import Model**

In [ ]:
MODEL_PATH = PROJECT_ROOT/"models"/"model.pkl"

# Use joblib.load instead of pickle.load
model = joblib.load(MODEL_PATH)

## **Apply Adaptive Conformal Prediction**

In [ ]:
# Predict calibration set
y_pred_cal = np.asarray(model.predict(n_periods=len(y_cal), X=X_cal))
y_pred_cal = pd.DataFrame(y_pred_cal)

In [ ]:
# Predict test set
y_pred_test = np.asarray(model.predict(n_periods=len(y_test), X=X_test))
y_pred_test = pd.DataFrame(y_pred_test)

In [ ]:
import numpy as np

# 1. Convert all pandas inputs into clean 1D NumPy arrays upfront
y_test = np.asarray(y_test).squeeze()
y_pred_test = np.asarray(y_pred_test).squeeze()

y_cal = np.asarray(y_cal).squeeze()
y_pred_cal = np.asarray(y_pred_cal).squeeze()

# 2. Compute 1D calibration residuals
cal_residuals = np.abs(y_cal - y_pred_cal)

# 3. Initialize tracker and results storage
tracker = ACITracker(alpha=0.05, gamma=0.01)
prediction_intervals = []

# 4. Online loop
for t in range(len(y_test)):
    y_true = float(y_test[t])
    pred = float(y_pred_test[t])

    # Step B: Get current alpha_t and compute interval margin
    current_alpha = tracker.alpha_t
    q_level = float(np.clip(1.0 - current_alpha, 0.0, 1.0))
    margin = float(np.quantile(cal_residuals, q_level))

    # Construct upper and lower prediction bounds
    lower_bound = pred - margin
    upper_bound = pred + margin

    if lower_bound < 0:
        lower_bound = 0
        
    prediction_intervals.append((lower_bound, upper_bound))

    # Step C: Check if true value was covered
    is_covered = lower_bound <= y_true <= upper_bound
    err_t = 0.0 if is_covered else 1.0

    # Step D: Update tracker
    tracker.update(err_t)

In [ ]:
lb = [interval[0] for interval in prediction_intervals]
ub = [interval[1] for interval in prediction_intervals]

In [ ]:
# 2. Unpack prediction intervals into 1D upper and lower bounds
intervals = np.array(prediction_intervals)
lower_bounds = intervals[:, 0].squeeze()
upper_bounds = intervals[:, 1].squeeze()

# 3. Create time steps based on the TEST set length (not y_cal)
time_steps = np.arange(len(y_test))

# 4. Create plot figure
plt.figure(figsize=(14, 6))

# Plot actual ground truth values
plt.plot(time_steps, y_test, label="Actual (y_true)", color="#0284c7", linewidth=1.5, alpha = 0.5)

# Plot model point predictions
plt.plot(
    time_steps,
    y_pred_test,
    label="Predicted Consumption",
    color='black',
    linewidth=1.2,
)


# Shade the prediction interval
plt.fill_between(
    time_steps,
    lb,  # or lb
    ub,  # or ub
    color="red",
    alpha=0.15,  
    label="Conformal Interval Prediction"
)


plt.title("Adaptive Conformal Prediction Intervals", fontsize=14, pad=10)
plt.xlabel("Time Step (t)", fontsize=11)
plt.ylabel("Value", fontsize=11)
plt.legend(loc="upper left", frameon=True)
plt.grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.show()